In [ ]:
# Load R magic extension for Python Jupyter kernel (Kaggle / Colab support)
try:
    %load_ext rpy2.ipython
except Exception as e:
    print("Note on rpy2 initialization:", e)

# Standalone 3-Tier Soft Probability Pipeline Transpilation to Pure C (`models/transpile_soft_pipeline_to_c.ipynb`)

This notebook exports all 4 trained LightGBM sub-models and transpile the entire **3-Tier Joint Soft Probability Product Triage Pipeline** into standalone, zero-dependency **pure C code** (`deploy/triage_pipeline.c` & `deploy/triage_pipeline.h`) using the **expanded 38-feature input vector** across all layers.

### Generated Embedded C Deliverables
- `deploy/triage_pipeline.h`: Public C Header API
- `deploy/triage_pipeline.c`: Complete Pure C Decision Tree & Soft Joint Probability Implementation
- `deploy/verify_c_pipeline`: Compiled Standalone Test Binary

In [ ]:
%%R
# ---------------------------------------------------------
# Step 1: Export All 4 LightGBM Model Trees & Scaler Parameters
# ---------------------------------------------------------
suppressPackageStartupMessages({
  library(jsonlite)
  library(lightgbm)
})
deploy_dir <- "../deploy"
if (!dir.exists(deploy_dir)) deploy_dir <- "deploy"
l1_obj  <- readRDS(file.path(deploy_dir, "lightgbm_layer1_esi1_model.rds"))
l2_obj  <- readRDS(file.path(deploy_dir, "rf_esi23_esi45_extreme_model.rds"))
l3a_obj <- readRDS(file.path(deploy_dir, "lightgbm_esi23_model.rds"))
l3b_obj <- readRDS(file.path(deploy_dir, "lightgbm_esi45_model.rds"))
lgb_l1_model  <- l1_obj$model
lgb_l2_model  <- l2_obj$model
lgb_l3a_model <- l3a_obj$model
lgb_l3b_model <- l3b_obj$model
scaler        <- l1_obj$scaler
dt_l1  <- lgb.model.dt.tree(lgb_l1_model)
dt_l2  <- lgb.model.dt.tree(lgb_l2_model)
dt_l3a <- lgb.model.dt.tree(lgb_l3a_model)
dt_l3b <- lgb.model.dt.tree(lgb_l3b_model)
scaler_dict <- list(means = as.list(scaler$means), sds = as.list(scaler$sds), cols = scaler$cols)
write(jsonlite::toJSON(dt_l1,  auto_unbox=TRUE), file.path(deploy_dir, "l1_trees.json"))
write(jsonlite::toJSON(dt_l2,  auto_unbox=TRUE), file.path(deploy_dir, "l2_trees.json"))
write(jsonlite::toJSON(dt_l3a, auto_unbox=TRUE), file.path(deploy_dir, "l3a_trees.json"))
write(jsonlite::toJSON(dt_l3b, auto_unbox=TRUE), file.path(deploy_dir, "l3b_trees.json"))
write(jsonlite::toJSON(scaler_dict, auto_unbox=TRUE), file.path(deploy_dir, "scaler.json"))
cat("All 4 Model Trees and Scaler parameters exported to deploy/*.json successfully!\n")

In [ ]:
# ---------------------------------------------------------
# Step 2: Transpile Decision Trees to Pure C Code (38 Features)
# ---------------------------------------------------------
import os
import json
import pandas as pd
import numpy as np
deploy_dir = '../deploy' if os.path.exists('../deploy') else 'deploy'
with open(os.path.join(deploy_dir, 'l1_trees.json')) as f: dt_l1 = json.load(f)
with open(os.path.join(deploy_dir, 'l2_trees.json')) as f: dt_l2 = json.load(f)
with open(os.path.join(deploy_dir, 'l3a_trees.json')) as f: dt_l3a = json.load(f)
with open(os.path.join(deploy_dir, 'l3b_trees.json')) as f: dt_l3b = json.load(f)
# UNIFORM 38-FEATURE LIST
feature_names = [
    'age', 'cc_breathingdifficulty', 'gender', 'triage_vital_hr', 'triage_vital_sbp', 'triage_vital_rr', 'triage_vital_o2',
    'pulse_min', 'resp_min', 'spo2_min', 'sbp_min', 'pulse_max', 'resp_max', 'spo2_max', 'sbp_max',
    'is_dyspnea_total', 'is_dyspnea_moderate', 'is_bradypnea', 'is_tachypnea', 'is_hypotension', 'is_hypertension',
    'is_bradycardia_total', 'is_bradycardia_moderate', 'is_tachycardia_total', 'is_tachycardia_moderate',
    'hr_range', 'rr_range', 'spo2_range', 'sbp_range',
    'shock_index', 'hr_mid_to_triage', 'sbp_mid_to_triage', 'rr_mid_to_triage', 'spo2_mid_to_triage',
    'rox_index', 'spo2_drop_ratio', 'hr_instability_ratio', 'bif'
]
def dt_to_dataframe(dt_json):
    return pd.DataFrame(dt_json)
df_l1  = dt_to_dataframe(dt_l1)
df_l2  = dt_to_dataframe(dt_l2)
df_l3a = dt_to_dataframe(dt_l3a)
df_l3b = dt_to_dataframe(dt_l3b)
def build_tree_c(df_tree, node_id, feat_list):
    row = df_tree[df_tree['node_index'] == node_id].iloc[0]
    if pd.isna(row['split_feature']):
        val = row['leaf_value']
        return f"return {val:.8f}f;"
    
    feat_name = row['split_feature']
    feat_idx  = feat_list.index(feat_name) if feat_name in feat_list else 0
    thresh    = row['threshold']
    l_child   = row['left_child']
    r_child   = row['right_child']
    
    left_code  = build_tree_c(df_tree, l_child, feat_list)
    right_code = build_tree_c(df_tree, r_child, feat_list)
    
    return f"if (x[{feat_idx}] <= {thresh:.8f}f) {{ {left_code} }} else {{ {right_code} }}"
def generate_model_func(df_model, func_name, feat_list):
    tree_ids = sorted(df_model['tree_index'].unique())
    code = f"static float {func_name}(const float x[38]) {{\n"
    code += "    float margin = 0.0f;\n"
    for t_id in tree_ids:
        df_t = df_model[df_model['tree_index'] == t_id]
        root_id = f"{t_id}-S0"
        tree_c = build_tree_c(df_t, root_id, feat_list)
        code += f"    margin += ({{ {tree_c} }});\n"
    code += "    return 1.0f / (1.0f + expf(-margin));\n}"
    return code
c_l1  = generate_model_func(df_l1,  "predict_layer1",  feature_names)
c_l2  = generate_model_func(df_l2,  "predict_layer2",  feature_names)
c_l3a = generate_model_func(df_l3a, "predict_layer3a", feature_names)
c_l3b = generate_model_func(df_l3b, "predict_layer3b", feature_names)
header_content = """#ifndef TRIAGE_PIPELINE_H
#define TRIAGE_PIPELINE_H

#ifdef __cplusplus
extern "C" {
#endif

// Raw Input Features (15 Values)
typedef struct {
    float age;
    float cc_breathingdifficulty;
    float gender;
    float triage_vital_hr;
    float triage_vital_sbp;
    float triage_vital_rr;
    float triage_vital_o2;
    float pulse_min;
    float resp_min;
    float spo2_min;
    float sbp_min;
    float pulse_max;
    float resp_max;
    float spo2_max;
    float sbp_max;
} TriageInput;

// 5-Class Output Probabilities and Predicted ESI Level
typedef struct {
    float probs[5]; // Index 0..4 maps to ESI 1..5
    int predicted_esi; // 1..5
} TriageOutput;

// Primary C Pipeline Entry Point
TriageOutput predict_triage(const TriageInput* input);

#ifdef __cplusplus
}
#endif

#endif // TRIAGE_PIPELINE_H
"""
c_source_content = f"""#include <math.h>
#include "triage_pipeline.h"

{c_l1}

{c_l2}

{c_l3a}

{c_l3b}

TriageOutput predict_triage(const TriageInput* in) {{
    TriageOutput out;
    float x[38];
    
    // Direct Raw Inputs
    x[0]  = in->age;
    x[1]  = in->cc_breathingdifficulty;
    x[2]  = in->gender;
    x[3]  = in->triage_vital_hr;
    x[4]  = in->triage_vital_sbp;
    x[5]  = in->triage_vital_rr;
    x[6]  = in->triage_vital_o2;
    x[7]  = in->pulse_min;
    x[8]  = in->resp_min;
    x[9]  = in->spo2_min;
    x[10] = in->sbp_min;
    x[11] = in->pulse_max;
    x[12] = in->resp_max;
    x[13] = in->spo2_max;
    x[14] = in->sbp_max;
    
    // Clinical Flags & Ranges
    x[15] = (in->triage_vital_o2 < 90.0f) ? 1.0f : 0.0f; // is_dyspnea_total
    x[16] = (in->triage_vital_o2 > 90.0f && in->triage_vital_o2 < 94.0f) ? 1.0f : 0.0f; // is_dyspnea_moderate
    x[17] = (in->triage_vital_rr < 10.0f) ? 1.0f : 0.0f; // is_bradypnea
    x[18] = (in->triage_vital_rr > 30.0f) ? 1.0f : 0.0f; // is_tachypnea
    x[19] = (in->triage_vital_sbp <= 90.0f) ? 1.0f : 0.0f; // is_hypotension
    x[20] = (in->triage_vital_sbp > 220.0f) ? 1.0f : 0.0f; // is_hypertension
    x[21] = (in->triage_vital_hr < 40.0f) ? 1.0f : 0.0f; // is_bradycardia_total
    x[22] = (in->triage_vital_hr > 40.0f && in->triage_vital_hr < 60.0f) ? 1.0f : 0.0f; // is_bradycardia_moderate
    x[23] = (in->triage_vital_hr > 150.0f) ? 1.0f : 0.0f; // is_tachycardia_total
    x[24] = (in->triage_vital_hr > 100.0f && in->triage_vital_hr < 150.0f) ? 1.0f : 0.0f; // is_tachycardia_moderate
    x[25] = in->pulse_max - in->pulse_min; // hr_range
    x[26] = in->resp_max - in->resp_min;   // rr_range
    x[27] = in->spo2_max - in->spo2_min;   // spo2_range
    x[28] = in->sbp_max - in->sbp_min;     // sbp_range
    
    // New Requested Advanced Clinical Indicators (9)
    x[29] = in->triage_vital_hr / ((in->triage_vital_sbp == 0.0f) ? 1.0f : in->triage_vital_sbp); // shock_index
    x[30] = in->triage_vital_hr - x[25]; // hr_mid_to_triage
    x[31] = in->triage_vital_sbp - x[28]; // sbp_mid_to_triage
    x[32] = in->triage_vital_rr - x[26]; // rr_mid_to_triage
    x[33] = in->triage_vital_o2 - x[27]; // spo2_mid_to_triage
    x[34] = in->triage_vital_o2 / ((in->triage_vital_rr == 0.0f) ? 1.0f : in->triage_vital_rr); // rox_index
    x[35] = x[27] / ((in->spo2_max == 0.0f) ? 1.0f : in->spo2_max); // spo2_drop_ratio
    x[36] = x[25] / (in->triage_vital_hr + 1.0f); // hr_instability_ratio
    x[37] = (in->triage_vital_rr / ((in->triage_vital_o2 == 0.0f) ? 1.0f : in->triage_vital_o2)) * 100.0f; // bif
    
    float p1  = predict_layer1(x);
    float p2  = predict_layer2(x);
    float p3a = predict_layer3a(x);
    float p3b = predict_layer3b(x);
    
    out.probs[0] = p1;
    out.probs[1] = (1.0f - p1) * p2 * p3a;
    out.probs[2] = (1.0f - p1) * p2 * (1.0f - p3a);
    out.probs[3] = (1.0f - p1) * (1.0f - p2) * p3b;
    out.probs[4] = (1.0f - p1) * (1.0f - p2) * (1.0f - p3b);
    
    int best_esi = 1;
    float max_p = out.probs[0];
    for (int k = 1; k < 5; k++) {{
        if (out.probs[k] > max_p) {{
            max_p = out.probs[k];
            best_esi = k + 1;
        }}
    }}
    out.predicted_esi = best_esi;
    return out;
}}
"""
with open(os.path.join(deploy_dir, 'triage_pipeline.h'), 'w') as f:
    f.write(header_content)
with open(os.path.join(deploy_dir, 'triage_pipeline.c'), 'w') as f:
    f.write(c_source_content)
print(f"Pure C Pipeline Source Transpiled with 38 Features: {os.path.join(deploy_dir, 'triage_pipeline.c')} & {os.path.join(deploy_dir, 'triage_pipeline.h')}")